Perfect 🚀 Let’s extend the framework with **LLM-based routing** instead of simple keyword rules.
We’ll use an LLM (OpenAI or Gemini) to decide whether the query should go to the **weather agent**, **pollution agent**, or **end**.

---

# 🔧 Update in Project

We add a new **router module** that queries an LLM to classify the prompt.

---

# 📂 Updated Structure

```
order_mgmt_framework/
│── main.py
│
├── config/
│   ├── settings.py
│
├── llm_router/
│   ├── __init__.py
│   ├── router.py   ← (NEW)
│
├── agents/
│   ├── parent_agent.py  ← updated to call LLM router
│   ├── agent_factory.py
│   ├── ...
```

---

# ⚙️ `config/settings.py` (update with LLM keys)

```python
# config/settings.py

AGENT_CONFIG = {
    "weather": {
        "llm": "openai",
        "tools": ["get_city_weather", "get_country_weather"],
        "mcp_servers": ["server_weather"]
    },
    "pollution": {
        "llm": "gemini",
        "tools": ["get_city_pollution", "get_country_pollution"],
        "mcp_servers": ["server_pollution"]
    },
    "parent": {
        "llm": "openai",  # or "gemini"
        "agents": ["weather", "pollution"],
        "protocol": "A2A",
        "features": ["RAG", "React", "ChainOfThought"]
    }
}

# API KEYS (set your own!)
OPENAI_API_KEY = "your-openai-key"
GEMINI_API_KEY = "your-gemini-key"
```

---

# 🧠 `llm_router/router.py`

```python
# llm_router/router.py
import os
from config.settings import OPENAI_API_KEY, GEMINI_API_KEY

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    import google.generativeai as genai
except ImportError:
    genai = None


def llm_route(prompt: str, llm_provider: str = "openai") -> str:
    """
    Uses LLM to classify query into 'weather', 'pollution', or 'end'.
    """

    classification_prompt = f"""
    You are a routing agent. 
    Classify the user query into one of: 'weather', 'pollution', 'end'.
    
    Query: {prompt}
    
    Respond ONLY with the label.
    """

    if llm_provider == "openai":
        if not OpenAI or not OPENAI_API_KEY:
            return "end"

        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": classification_prompt}],
            max_tokens=5
        )
        return resp.choices[0].message.content.strip().lower()

    elif llm_provider == "gemini":
        if not genai or not GEMINI_API_KEY:
            return "end"

        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel("gemini-pro")
        resp = model.generate_content(classification_prompt)
        return resp.text.strip().lower()

    return "end"
```

---

# 🧭 `agents/parent_agent.py` (updated)

```python
# agents/parent_agent.py
from config.settings import AGENT_CONFIG
from llm_router.router import llm_route

def parent_router(prompt: str) -> str:
    """Uses LLM instead of keyword rules for routing"""
    parent_cfg = AGENT_CONFIG["parent"]
    llm_provider = parent_cfg.get("llm", "openai")

    decision = llm_route(prompt, llm_provider)

    if decision not in ["weather", "pollution", "end"]:
        return "end"
    return decision
```

---

# 🚀 `main.py` (unchanged logic, just benefits from new router)

```python
# main.py
from langgraph.graph import StateGraph, START, END
from agents.agent_factory import AgentFactory
from agents.parent_agent import parent_router

def build_graph():
    factory = AgentFactory()
    weather_agent = factory.get_agent("weather")
    pollution_agent = factory.get_agent("pollution")

    builder = StateGraph(dict)

    builder.add_node("parent_router", lambda state: {"next": parent_router(state["prompt"])})
    builder.add_node("weather", lambda state: {"result": weather_agent(state)})
    builder.add_node("pollution", lambda state: {"result": pollution_agent(state)})

    builder.add_edge(START, "parent_router")
    builder.add_conditional_edges(
        "parent_router",
        lambda out: out["next"],
        {"weather": "weather", "pollution": "pollution", "end": END}
    )
    builder.add_edge("weather", END)
    builder.add_edge("pollution", END)

    return builder.compile()

def main():
    graph = build_graph()

    prompts = [
        {"prompt": "What's the weather in Paris?", "city": "Paris"},
        {"prompt": "Check AQI level in Delhi", "city": "Delhi"},
        {"prompt": "Tell me a joke"}  # should go to "end"
    ]

    for p in prompts:
        print(f"\n📝 Prompt: {p['prompt']}")
        result = graph.invoke(p)
        print("🤖 Response:", result.get("result"))

if __name__ == "__main__":
    main()
```

---

✅ Now the **parent agent** uses **LLM-based routing** instead of keywords.

* If user says *“What’s the weather in Paris?”* → goes to **Weather Agent**
* If user says *“Check AQI in Delhi”* → goes to **Pollution Agent**
* If user says *“Tell me a joke”* → goes to **End**

---

Do you want me to also **add a fallback rule-based router** in case the LLM is unavailable (e.g. offline/local dev)?
